In [2]:
from pyspark.sql import SparkSession
from pyspark import SparkConf

conf = SparkConf()
conf.set('spark.app.name', 'PySpark UDF')
conf.set('spark.master', 'local[*]')

spark = SparkSession.builder\
        .config(conf = conf)\
        .getOrCreate()

In [ ]:
columns = ["Seqno","Name"]
data = [("1", "john jones"),
        ("2", "tracey smith"),
        ("3", "amy sanders")]

df = spark.createDataFrame(data, schema = columns)

df.show(truncate = False)

+-----+------------+
|Seqno|Name        |
+-----+------------+
|1    |john jones  |
|2    |tracey smith|
|3    |amy sanders |
+-----+------------+



In [4]:
import pyspark.sql.functions as f

upper_func = f.udf(lambda x: x.upper())

df.withColumn('Upper Name', upper_func(f.col('Name'))).show()

+-----+------------+------------+
|Seqno|        Name|  Upper Name|
+-----+------------+------------+
|    1|  john jones|  JOHN JONES|
|    2|tracey smith|TRACEY SMITH|
|    3| amy sanders| AMY SANDERS|
+-----+------------+------------+



In [6]:
from pyspark.sql.types import StringType

def custom_upper_func(s):
    return s.upper()

upper_func = f.udf(custom_upper_func, StringType())

df.withColumn('Upper Name', upper_func(f.col('Name'))).show()

+-----+------------+------------+
|Seqno|        Name|  Upper Name|
+-----+------------+------------+
|    1|  john jones|  JOHN JONES|
|    2|tracey smith|TRACEY SMITH|
|    3| amy sanders| AMY SANDERS|
+-----+------------+------------+



In [ ]:
df.select(f.col('Name'), upper_func(f.col('Name'))).show()
# df.select(f.col('Name'), upper_func(f.col('Name').alias('Upper Name'))).show()

+------------+-----------------------+
|        Name|custom_upper_func(Name)|
+------------+-----------------------+
|  john jones|             JOHN JONES|
|tracey smith|           TRACEY SMITH|
| amy sanders|            AMY SANDERS|
+------------+-----------------------+



In [9]:
@f.udf(StringType())
def upper_func_decorator(s):
    return s.upper()

df.select(f.col('Name'), upper_func_decorator(f.col('Name'))).show()

+------------+--------------------------+
|        Name|upper_func_decorator(Name)|
+------------+--------------------------+
|  john jones|                JOHN JONES|
|tracey smith|              TRACEY SMITH|
| amy sanders|               AMY SANDERS|
+------------+--------------------------+



In [ ]:
# need pyarrow
import pandas as pd

@f.pandas_udf(StringType())
def upper_func_pandas(s: pd.Series) -> pd.Series:
    return s.str.upper()

upper_func = spark.udf.register('upper_func_pandas', upper_func_pandas)

spark.sql("SELECT upper_func_pandas('abcd')").show()

+-----------------------+
|upper_func_pandas(abcd)|
+-----------------------+
|                   ABCD|
+-----------------------+



In [15]:
df.select('name', upper_func('name')).show()

+------------+----------------+
|        name|upper_func(name)|
+------------+----------------+
|  john jones|      JOHN JONES|
|tracey smith|    TRACEY SMITH|
| amy sanders|     AMY SANDERS|
+------------+----------------+

